# Teacher training — own R50 (full COCO, 36 epochs)

Produces the teacher checkpoint every distillation run in this study distils
from. Final result: **val2017 mAP 0.142**, plateauing over the last five
epochs. Wall clock ~11 h on a Colab A100.

## Read this before re-running

An earlier attempt used `train_kd.py`'s default `lr_head=1e-3` and reached
only **0.027** mAP — while the training loss fell normally the whole way.
Every predicted class probability sat below 0.12; the model learned to be
uniformly unconfident. Dropping to `lr_head=1e-4` / `lr_backbone=1e-5` gave
0.142 under otherwise identical settings. **The LR flags below are not
optional.** See `docs/` for the full diagnosis.

## Settings and why

| Setting | Value | Reason |
|---|---|---|
| config | `rtdetr_r50vd_coco.yml` | `--teacher-source own` builds the teacher from *this* file; training with another architecture makes the checkpoint unloadable later |
| `--img-size` | 512 | Teacher and student process the same batch during KD, so the teacher must be strong at the ablation's resolution |
| `--batch-size` | 16, accum 1 | A100 80 GB |
| `--lr-head` / `--lr-backbone` | 1e-4 / 1e-5 | See above |
| data | full COCO 118K | The students train on a 30K subset; the teacher sees everything |

COCO goes on the Colab **local disk** — reading 118K small files through
mounted Drive would dominate epoch time. Checkpoints go the other way, to
Drive, so a dropped session costs nothing.


## 1 — Runtime

In [ ]:
gpu = !nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print(gpu[0])
assert 'A100' in gpu[0], f'Expected an A100, got: {gpu[0]}'
print('OK — A100 confirmed.')

## 2 — Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RUNS    = '/content/drive/MyDrive/rtdetr_runs'
OUT_DIR = f'{RUNS}/teacher_r50_lr1e4'
LOG     = f'{RUNS}/teacher_r50_lr1e4.log'
os.makedirs(OUT_DIR, exist_ok=True)
print('checkpoints ->', OUT_DIR)

## 3 — Repo

In [ ]:
%cd /content
if not os.path.exists('/content/rt-detr-kd'):
    !git clone --recurse-submodules https://github.com/umutonuryasar/rt-detr-kd
%cd /content/rt-detr-kd
!git pull --ff-only
!git log --oneline -3

## 4 — Dependencies

Not `uv sync` here: Colab ships a torch build matched to its own CUDA driver
and replacing it risks a mismatch. Install only what is missing.

In [ ]:
!pip install -q pycocotools scipy pyyaml tensorboard
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available()

## 5 — Full COCO to local disk (~20 min)

Re-run after any session restart — local disk does not survive.

In [ ]:
if not os.path.exists('/content/coco/train2017'):
    !bash scripts/download_coco_full.sh /content/coco
else:
    print('COCO already present.')

n_train = len(os.listdir('/content/coco/train2017'))
n_val   = len(os.listdir('/content/coco/val2017'))
print(f'train2017: {n_train}   val2017: {n_val}')
assert n_train == 118287, f'expected 118287, got {n_train} — download incomplete'
assert n_val == 5000, f'expected 5000, got {n_val} — download incomplete'
print('OK — dataset complete.')

## 6 — Train

Resumes automatically from `checkpoint_latest.pth` if the session dropped.
Safe to re-run: re-do cells 1–5 first (local disk is wiped), then this one.

### Check after epoch 1

- LR peaks near **1.00e-04** around scheduler step 500, then decays. Still
  climbing at the end of epoch 1 means the schedule horizon is wrong — stop.
- Epoch wall clock 15–25 min. Multiply by 36 to plan sessions.
- Epoch-1 mAP small but non-zero (this run: 0.0102). Exactly 0.0000 after a
  full-data epoch means something is wrong.
- No `lr_scheduler.step() before optimizer.step()` warning.
- `checkpoint_latest.pth` appears in the Drive folder.

### Abort criterion

If val mAP is still ~0 at the end of epoch 3 and the loss has stopped falling,
kill it — a falling loss does **not** rule out the collapse described at the
top of this notebook.

In [ ]:
latest = f'{OUT_DIR}/checkpoint_latest.pth'
resume = f'--student-weights {latest}' if os.path.exists(latest) else ''
print(f'RESUMING from {latest}\n' if resume else 'FRESH START\n')

cmd = (
    'python tools/train_kd.py '
    '--student-cfg configs/rtdetr_r50vd_coco.yml '
    '--kd-type none '
    '--epochs 36 --batch-size 16 --accumulate-steps 1 --img-size 512 '
    '--lr-head 1e-4 --lr-backbone 1e-5 '
    '--coco-train /content/coco/train2017 '
    '--train-ann /content/coco/annotations/instances_train2017.json '
    '--coco-val /content/coco/val2017 '
    '--val-ann /content/coco/annotations/instances_val2017.json '
    '--num-workers 8 --seed 42 --use-amp --save-every 5 '
    f'--output-dir {OUT_DIR} {resume} '
    f'2>&1 | tee -a {LOG}'
)
print(cmd, '\n' + '-' * 70 + '\n')
!{cmd}

## 7 — Result

In [ ]:
!ls -la {OUT_DIR}
print('\n--- mAP history ---')
!grep -E 'mAP|Training complete' {LOG} | tail -20

## Next

`checkpoint_best.pth` in this folder is the teacher for every ablation run.
Continue in `ablation_colab.ipynb`, which calibrates λ against this checkpoint
and runs the campaign.